# 6.0 - GAN Generative Replay

The far-left end of the memory curve: instead of storing exemplars, store only a per-class
prototype (mu, sigma) and let a frozen generator synthesise features on demand. That is
about 2 KB per class, versus ~2 MB for SmartReplay.

This notebook works on the head's 256-d embedding space (the LSTM output), not the 2048-d
frame features. It trains the generator, then **shows how it generates embeddings**: a
t-SNE of real vs generated features, and a check that a classifier trained only on
generated features can still recognise real ones.

It uses `train_gan_real`, the corrected loop. The original `train_gan` fed the class MEAN
as the "real" sample, so the discriminator only ever saw one vector per class and the
generator collapsed to x = mu. Here the discriminator sees the real per-clip features, so
the generator has to match the real spread, not just its centre.

The zero-shot half of GIL is dropped: it learns a text-to-prototype map from 10 pairs and
cannot extrapolate to unseen names at this scale.

## **Connect Colab**

In [1]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

## **Download DATASET**

Drive only - the cache and the base head are on Drive, no video needed.

In [2]:
from src.bootstrap import setup
setup(drive=True, dataset=False)

Drive mounted.
ULAL_DATA_ROOT     = /content/UCF101
ULAL_DRIVE_PROJECT = /content/drive/MyDrive/apai
Setup complete — no config.py edit, no kernel restart.


In [3]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

ULAL_ROOT : /content/ulal
Device    : cuda
GPU       : Tesla T4 (15.6 GB)
Classes   : base=10 t1=10 t2=10 t3=10 t4=10


42

## **Load Cache and Base Head**

The base head is the same `head_base.pt` the continual-learning notebooks use, so the
feature space is identical. If it is not on Drive yet it is trained here on task 0.

In [4]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
check_cache_labels(META, ALL_CLASSES_LIST)

class_to_idx = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}
print(f"base classes: {len(cfg.SELECTED_CLASSES)}")

Loaded 9 tensors, 526 MB in RAM
  per clip: (16, 2048)
base classes: 10


In [5]:
head = load_or_train_base_head(
    f"{cfg.CKPT_DIR}/head_base.pt", CACHE, LSTM_STATE, len(cfg.SELECTED_CLASSES), device,
)
print(f"head embedding dim: {cfg.TEACHER_HIDDEN}")

Loaded base head <- /content/drive/MyDrive/apai/checkpoints/head_base.pt
head embedding dim: 256


## **Feature Space (256-d)**

Run the base head over the cached frames and keep its 256-d output per clip. These real
features are what the generator will learn to reproduce. Train split trains the GAN, test
split checks it later.

In [6]:
real_feats, real_labels = head_embeddings(head, CACHE, [0], device, "train")
test_feats, test_labels = head_embeddings(head, CACHE, [0], device, "test")

print(f"real train feats: {tuple(real_feats.shape)}   labels {tuple(real_labels.shape)}")
print(f"real test  feats: {tuple(test_feats.shape)}")

real train feats: (953, 256)   labels (953,)
real test  feats: (212, 256)


## **Semantic Embeddings**

Each class name is embedded with all-MiniLM-L6-v2 (384-d). The discriminator and the
mutual-information term are conditioned on this vector. CamelCase names are split into
words first so "ApplyEyeMakeup" reads as "Apply Eye Makeup".

In [7]:
!pip install -q sentence-transformers

In [ ]:
import re
from sentence_transformers import SentenceTransformer

class_sentences = [re.sub(r"(?<!^)(?=[A-Z])", " ", c) for c in cfg.SELECTED_CLASSES]
print(class_sentences)

sbert  = SentenceTransformer("all-MiniLM-L6-v2", device=str(device))
sem_np = sbert.encode(class_sentences, convert_to_numpy=True, normalize_embeddings=True)

print(f"semantic embeddings: {sem_np.shape}")

## **Class Prototypes**

For each base class, store the mean and std of its real features, plus its semantic
vector. This is the whole memory footprint of GIL - a few KB per class, no clips.

In [ ]:
FEAT_DIM = cfg.TEACHER_HIDDEN

class_stats = {}
for i in range(len(cfg.SELECTED_CLASSES)):
    mask = real_labels == i
    fc   = real_feats[mask]
    class_stats[i] = {
        "mu":    fc.mean(0),
        "sigma": fc.std(0).clamp_min(1e-6),
        "sem":   torch.tensor(sem_np[i], dtype=torch.float32),
    }

bytes_per_class = 2 * FEAT_DIM * class_stats[0]["mu"].element_size()
print(f"stored per class: mu+sigma = {bytes_per_class} bytes ({bytes_per_class/1e3:.1f} KB)")

## **Train the Generator**

WGAN-GP with a gradient penalty, an auxiliary classifier on the synthetic features, and a
mutual-information term back to the semantic space. `train_gan_real` sees the real per-clip
features, so it cannot win by collapsing to the class mean.

In [ ]:
gen  = FeatureGenerator(feat_dim=FEAT_DIM, noise_dim=cfg.NOISE_DIM).to(device)
disc = Discriminator(feat_dim=FEAT_DIM, sem_dim=cfg.SEM_DIM).to(device)
proj = ProjectionHead(feat_dim=FEAT_DIM, sem_dim=cfg.SEM_DIM).to(device)

print(f"generator params: {sum(p.numel() for p in gen.parameters()):,}")

In [ ]:
set_seed(cfg.SEED)

gan_history = train_gan_real(
    gen, disc, proj, real_feats, real_labels, class_stats, device,
    epochs=cfg.GAN_EPOCHS, lam1=cfg.LAM1, lam2=cfg.LAM2, alpha=cfg.ALPHA_GP,
    lr=cfg.GAN_LR, batch_size=cfg.BATCH_SIZE_GAN, n_critic=cfg.N_CRITIC,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
eps = range(1, len(gan_history["d_loss"]) + 1)

ax1.plot(eps, gan_history["d_loss"], label="D")
ax1.plot(eps, gan_history["g_loss"], label="G (adv)")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title("WGAN-GP losses"); ax1.legend()

ax2.plot(eps, gan_history["cls_loss"], color="green")
ax2.set_xlabel("epoch"); ax2.set_ylabel("CE"); ax2.set_title("Aux-classifier loss on synthetic feats")

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/gil_training.png", dpi=150, bbox_inches="tight")
plt.show()

## **Generate Embeddings**

Ask the frozen generator for J synthetic features per class, from (mu, sigma) plus fresh
noise each time. Nothing but the prototypes was stored.

In [ ]:
J = cfg.J_SYNTH
gen.eval()

synth_list, synth_lab = [], []
with torch.no_grad():
    for i in range(len(cfg.SELECTED_CLASSES)):
        mu    = class_stats[i]["mu"].unsqueeze(0).expand(J, -1).to(device)
        sigma = class_stats[i]["sigma"].unsqueeze(0).expand(J, -1).to(device)
        xh    = gen(mu, sigma).cpu()
        synth_list.append(xh)
        synth_lab.extend([i] * J)

synth_feats  = torch.cat(synth_list)
synth_labels = torch.tensor(synth_lab)
print(f"generated: {tuple(synth_feats.shape)}  ({J} per class)")

## **Real vs Generated**

The picture that answers "how does it generate the embeddings". A t-SNE of real (circles)
and generated (crosses) features. If the generation is good the crosses land inside the
matching real cluster and spread across it. If they pile onto a single point per class,
that is mode collapse.

In [ ]:
n_per = J
real_sub, real_sub_lab = [], []
for i in range(len(cfg.SELECTED_CLASSES)):
    idx = (real_labels == i).nonzero(as_tuple=True)[0][:n_per]
    real_sub.append(real_feats[idx])
    real_sub_lab.extend([i] * len(idx))
real_sub     = torch.cat(real_sub)
real_sub_lab = torch.tensor(real_sub_lab)

X      = torch.cat([real_sub, synth_feats]).numpy()
origin = np.array([0] * len(real_sub) + [1] * len(synth_feats))
labs   = np.concatenate([real_sub_lab.numpy(), synth_labels.numpy()])

Xp = PCA(n_components=50, random_state=cfg.SEED).fit_transform(X) if X.shape[1] > 50 else X
xy = TSNE(n_components=2, random_state=cfg.SEED, init="pca", learning_rate="auto",
          perplexity=30).fit_transform(Xp)

print(f"projected {len(xy)} points")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
cmap = plt.get_cmap("tab10")

for i in range(len(cfg.SELECTED_CLASSES)):
    r = (labs == i) & (origin == 0)
    g = (labs == i) & (origin == 1)
    ax1.scatter(xy[r, 0], xy[r, 1], color=cmap(i % 10), marker="o", s=16, alpha=0.5)
    ax1.scatter(xy[g, 0], xy[g, 1], color=cmap(i % 10), marker="x", s=28)
ax1.set_title("By class: real (o) vs generated (x)")

ax2.scatter(xy[origin == 0, 0], xy[origin == 0, 1], s=16, alpha=0.5, label="real")
ax2.scatter(xy[origin == 1, 0], xy[origin == 1, 1], s=16, alpha=0.5, label="generated")
ax2.legend()
ax2.set_title("Real vs generated distribution")

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/gil_real_vs_generated.png", dpi=150, bbox_inches="tight")
plt.show()

Per-class cosine similarity between the real centroid and the generated centroid. Near 1.0
means the generator centres each class correctly. Read it together with the plot: a high
cosine but a collapsed cloud would still be a bad generator.

In [ ]:
print(f"{'class':<22}{'cos(real, gen) centroid':>26}")
print("-" * 48)
for i in range(len(cfg.SELECTED_CLASSES)):
    real_c = class_stats[i]["mu"]
    gen_c  = synth_feats[synth_labels == i].mean(0)
    cos    = F.cosine_similarity(real_c.unsqueeze(0), gen_c.unsqueeze(0)).item()
    print(f"{cfg.SELECTED_CLASSES[i]:<22}{cos:>26.3f}")

## **Does the Generation Carry the Classes?**

The honest test of generative replay: train a fresh classifier on ONLY the generated
features, then measure it on REAL test features. Compare against the same classifier
trained on real features. If the generated-only number is close, the generator is a usable
stand-in for stored data.

In [ ]:
set_seed(cfg.SEED)
gen_clf = GILClassifier(feat_dim=FEAT_DIM, num_classes=len(cfg.SELECTED_CLASSES)).to(device)
gen_loader = DataLoader(TensorDataset(synth_feats, synth_labels), batch_size=64, shuffle=True)
fine_tune_last2(gen_clf, gen_loader, epochs=cfg.FT_EPOCHS, lr=cfg.FT_LR, device=device)

with torch.no_grad():
    gen_pred = gen_clf(test_feats.to(device)).argmax(1).cpu()
gen_acc = (gen_pred == test_labels).float().mean().item()

In [ ]:
set_seed(cfg.SEED)
real_clf = GILClassifier(feat_dim=FEAT_DIM, num_classes=len(cfg.SELECTED_CLASSES)).to(device)
real_loader = DataLoader(TensorDataset(real_feats, real_labels), batch_size=64, shuffle=True)
fine_tune_last2(real_clf, real_loader, epochs=cfg.FT_EPOCHS, lr=cfg.FT_LR, device=device)

with torch.no_grad():
    real_pred = real_clf(test_feats.to(device)).argmax(1).cpu()
real_acc = (real_pred == test_labels).float().mean().item()

In [ ]:
print(f"trained on REAL features      -> real test acc: {real_acc:.2%}")
print(f"trained on GENERATED features -> real test acc: {gen_acc:.2%}")
print(f"gap (cost of replacing data with the generator): {real_acc - gen_acc:.2%}")

In [ ]:
plot_confusion_matrix(
    test_labels.numpy(), gen_pred.numpy(),
    num_classes  = len(cfg.SELECTED_CLASSES),
    classes_list = cfg.SELECTED_CLASSES,
    name         = "trained on generated, tested on real",
    save_path    = f"{cfg.RESULTS_DIR}/gil_cm_generated",
)

## **Save**

Metrics and the generator, so a later notebook can reuse the frozen generator as replay.

In [ ]:
gil_result = {
    "stage":           "6_gil_generative_replay",
    "classes":         cfg.SELECTED_CLASSES,
    "feat_dim":        FEAT_DIM,
    "bytes_per_class": int(bytes_per_class),
    "J_synth":         cfg.J_SYNTH,
    "real_test_acc":   float(real_acc),
    "gen_test_acc":    float(gen_acc),
    "gap":             float(real_acc - gen_acc),
}

os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage6_gil.json", "w") as f:
    json.dump(gil_result, f, indent=2, default=float)

os.makedirs(cfg.GIL_CKPT_DIR, exist_ok=True)
torch.save(gen.state_dict(), f"{cfg.GIL_CKPT_DIR}/generator.pth")

print(f"Saved -> {cfg.RESULTS_DIR}/stage6_gil.json")
print(f"Saved -> {cfg.GIL_CKPT_DIR}/generator.pth")